# Next step: step 6a, timing and orbits
**What it adds:** two read-only views of *when* a run happens, and the rename of `simulate`:

| Function | Returns | Speed |
|---|---|---|
| `sim.get_timing(params, config)` | the start time, step count, whether SPAM's −5 fallback fired, and the closest approach SPAM's search found | instant |
| `sim.simulate_orbit(params, config)` | the secondary's center at every step, shape `(n_steps + 1, 7)`: x, y, z, vx, vy, vz, t | about 0.05 s |
| `sim.simulate_particles(params, config, seed)` | **renamed from `simulate`**, otherwise unchanged | about 2 s |

**How SPAM picks the start** (`getTStart`, details in `docs/spam_physics.md`, "The start time"):
1. Run the two centers **backward** from now (t = 0) to t = −30, and keep their **closest** point over that
   whole window.
2. Start a lead of `10 × rout1 / speed at closest approach` before it (or near the last apocenter, for a bound
   orbit).
3. If that's earlier than **−12** (or within a step of 0), silently start at **−5** instead.

One time unit is about 87 Myr, so −12 is about 1 Gyr ago.

**How it connects:**
- **Below, in JSPAM:** `create_collision` now *keeps* what the search found and whether the fallback fired; the
  start it picks is unchanged. `simr_interface` gets `get_timing` and `run_orbit`, and `jspam` gets `timing()`
  and `run_orbit()`.
- **Next, step 6b:** expose the search's fixed numbers (the −30 window, the −12 cap, the −5 fallback, the 10 in
  the lead) or a fixed start time. This step's plots are how we decide which matter.
- **Step 7 (many endings):** saving particles at several times needs step counts, which `get_timing` provides.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `fortran/parameters_module.f90` and `init_module.f90`:** two new module variables, filled in by
`create_collision` right where it chooses the start:
```fortran
real (kind=8), dimension(4) :: tstart_search = 0.0d0   ! getTStart's output: start, distance, speed, time of closest
logical :: tstart_fallback = .false.                    ! true when the start was replaced by -5

tminVals = getTStart(rv4min, ...)
tmpT = tminVals(1)
tstart_search = tminVals    ! SIMR: kept for reporting
tstart_fallback = .false.   ! SIMR
if ( tmpT < -12.0 ) then
  tmpT = -5
  tstart_fallback = .true.  ! SIMR      (the same in the "within one step of 0" branch)
endif
```
**JSPAM, `fortran/simr_interface.f90`:** both set up a run with **0 particles**, so only the two centers exist:
```fortran
subroutine get_timing(collision_param, potential, t_start, n_steps, start_fallback, search)
  call setup_run(collision_param, potential, 0, 0, 0.0d0, 0.0d0, 0)
  t_start = tstart
  n_steps = int((tend - tstart) / h) + 2      ! the same count run_collision steps
  ...
subroutine run_orbit(collision_param, potential, n_steps, orbit)
  call setup_run(collision_param, potential, 0, 0, 0.0d0, 0.0d0, 0)
  orbit(1,1:6) = x0(n,1:6)                    ! the secondary is x0's last row
  orbit(1,7) = time
  do step = 1, n_steps
    call TAKE_A_STEP
    orbit(step+1,1:6) = x0(n,1:6)
    orbit(step+1,7) = time
  enddo
```
(`x0(n,:)`, not `x0(n+1,:)`: placing the secondary does `n = n + 1`. The first version read one column over,
and the tests caught it.)

**JSPAM, `src/jspam/__init__.py`:** `timing(params, ...)` turns the Fortran outputs into a dict, and
`run_orbit(params, n_steps=None, ...)` asks `timing` for the step count when it isn't given.

**`sim.py`:** `simulate` is now `simulate_particles`, plus:
```python
def get_timing(params, config=SimulatorConfig()) -> dict:
    return jspam.timing(_spam_params(params), **_orbit_args(config))

def simulate_orbit(params, config=SimulatorConfig()) -> np.ndarray:
    return jspam.run_orbit(_spam_params(params), **_orbit_args(config))

def _orbit_args(config):   # only the mass model moves the centers; particles and heat don't
    return {"mass_model": _mass_model(config), "normalization": config.normalization}
```
**Tests:** the orbit starts where `initial_state` puts the secondary and ends exactly where `simulate_particles`
leaves it; its closest point matches `get_timing`'s; the lead formula holds; particles and heat don't change
the orbit, the mass model does; and the two failure cases below.

In [ ]:
%matplotlib inline
import time
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
config = sim.SimulatorConfig()
MYR = 87  # about 87 Myr per SPAM time unit
np.set_printoptions(suppress=True)  # plain numbers, not 1.0e-03

## 1. When does the run start?

In [ ]:
timing = sim.get_timing(params, config)
for name, value in timing.items():
    print(f"{name:17s} {value}")

rout1 = params[sim.PARAM_INDEX["rout1"]]
lead = 10 * rout1 / timing["closest_speed"]
print(f"\nlead = 10 x rout1 / speed = 10 x {rout1:.3f} / {timing['closest_speed']:.3f} = {lead:.3f}")
print(f"t_closest - lead = {timing['t_closest']:.3f} - {lead:.3f} = {timing['t_closest'] - lead:.3f}")
print(f"so the run starts about {-timing['t_start'] * MYR:.0f} Myr ago, "
      f"{(timing['t_closest'] - timing['t_start']) * MYR:.0f} Myr before the closest approach")

## 2. The orbit
One row per step, from the start to 1–2 steps past now. The last row is the observed position: the model's own
x, y, z, vx, vy, vz (to within the 1–2 extra steps).

In [ ]:
start_clock = time.perf_counter()
orbit = sim.simulate_orbit(params, config)
print(f"shape {orbit.shape}, in {time.perf_counter() - start_clock:.3f} s")
print("first row:", orbit[0].round(3))
print("last row: ", orbit[-1].round(3))
print("the model:", params[:6])

## 3. The orbit, drawn
Left: the secondary's path around the primary (at the origin), seen on the sky (x, y). Right: the distance
between the centers over time, with SPAM's −12 cap and −5 fallback marked.

In [ ]:
t = orbit[:, 6]
separation = np.linalg.norm(orbit[:, :3], axis=1)
closest = separation.argmin()

fig, (ax_xy, ax_sep) = plt.subplots(1, 2, figsize=(15, 6))
ax_xy.plot(orbit[:, 0], orbit[:, 1], color="C1", label="secondary's path")
ax_xy.plot(0, 0, "k+", ms=14, label="primary (origin)")
ax_xy.plot(*orbit[0, :2], "go", label=f"start, t = {t[0]:.2f}")
ax_xy.plot(*orbit[closest, :2], "rs", label=f"closest, t = {t[closest]:.2f}")
ax_xy.plot(*orbit[-1, :2], "k^", label="now")
ax_xy.set_aspect("equal")
ax_xy.set_xlabel("x")
ax_xy.set_ylabel("y")
ax_xy.legend()

ax_sep.plot(t, separation, color="C1")
ax_sep.axvline(t[0], color="g", ls="--", label="start")
ax_sep.axvline(t[closest], color="r", ls="--", label="closest approach")
ax_sep.axvline(-12, color="0.5", ls=":", label="−12 cap")
ax_sep.axvline(-5, color="0.5", ls="-.", label="−5 fallback")
ax_sep.set_xlim(-13, 0.5)
ax_sep.set_xlabel("time (SPAM units, now = 0)")
ax_sep.set_ylabel("distance between centers")
ax_sep.legend()
plt.show()

## 4. The orbit over a particle run
The centers don't feel the particles, so the orbit is the same for any particles or seed, and its last row is
exactly the particle run's last row:

In [ ]:
N = 2000
particles = sim.simulate_particles(params, replace(config, n1=N, n2=N), seed=1)
print("orbit's last row == particle run's secondary center:", np.array_equal(orbit[-1, :6], particles[-1]))

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(particles[:N, 0], particles[:N, 1], s=1, label="primary disk")
ax.scatter(particles[N:2 * N, 0], particles[N:2 * N, 1], s=1, label="secondary disk")
ax.plot(orbit[:, 0], orbit[:, 1], "k", lw=1, label="secondary's path")
ax.plot(0, 0, "k+", ms=14)
ax.set_aspect("equal")
ax.legend(markerscale=5)
plt.show()

## 5. When the search goes wrong
The mass model changes the force, so it changes the orbit's history (never its end: every run ends at the
observed position). `m_halo` with `"total"` normalization changes nothing here, because the centers never come
within r = 2, where all the galaxy's mass is inside. `r_scale` changes the force everywhere:

In [ ]:
cases = {"default": config, "m_halo 11.6": replace(config, m_halo=11.6)}
cases.update({f"r_scale {v:g}": replace(config, r_scale=v) for v in [5.0, 8.0, 9.0, 12.0, 20.0]})

print(f"{'':12s} {'t_start':>8s} {'found':>8s} {'t_closest':>9s} {'distance':>8s}  fallback")
for label, cfg in cases.items():
    tm = sim.get_timing(params, cfg)
    print(f"{label:12s} {tm['t_start']:8.2f} {tm['t_start_found']:8.2f} {tm['t_closest']:9.2f} "
          f"{tm['closest_distance']:8.2f}  {tm['start_fallback']}")

Two problems show up:
- **A weaker force (`r_scale` 12, 20) works as intended:** the closest approach is still near −3, and the start
  moves earlier.
- **A slightly stronger force (`r_scale` 8, 9) makes the search pick an older pass.** It keeps the closest point
  anywhere back to −30, and now finds one near −20 to −23. Its start is then before −12, so SPAM falls back to −5.
  The run is only 5 units long and starts after the pass the search found.

The orbit only covers the run itself, from `t_start`, so the older pass isn't drawn:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label in ["default", "r_scale 9", "r_scale 12", "r_scale 20"]:
    path = sim.simulate_orbit(params, cases[label])
    ax.plot(path[:, 6], np.linalg.norm(path[:, :3], axis=1), label=label)
ax.axvline(-5, color="0.5", ls="-.", label="−5 fallback")
ax.set_xlabel("time (SPAM units, now = 0)")
ax.set_ylabel("distance between centers")
ax.legend()
plt.show()